<a href="https://colab.research.google.com/github/Eno-stack/MLU/blob/main/lab4_langgraph.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Building LLM Applications with LangChain + Agents
**Tech Launchpad · City Colleges of Chicago · Instructor: Lance Muwayi**

## Lab 4 · LangGraph: a plan → research → draft → critique → revise pipeline
**Session 4 · Thu 11/12 · 8:20–8:50**

You will build a five-node graph with a bounded critique loop, compile it with a checkpointer, stream its progress, and inspect its history.

**Done when** the graph produces a final draft after at most 3 critique rounds and `get_state_history` shows the checkpoints (Steps 1–5; Step 6 is optional).

In [15]:
# ===== Shared setup: every lab starts with this cell. Works on Google Colab (default) and locally. =====
import os, sys, subprocess, pathlib, shutil

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    # 1) packages (quiet; ~60 s the first time per session)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U",
                    "langchain>=1.3", "langgraph>=1.2", "langchain-text-splitters",
                    "langchain-google-genai", "pandas", "python-dotenv", "fastapi", "httpx"], check=True)
    # 2) course data (clone once per session)
    if not pathlib.Path("data").exists():
        if pathlib.Path("course").exists():
            shutil.rmtree("course")
        try:
            subprocess.run(["git", "clone", "-q", "https://github.com/Eno-stack/healthcare-appeal-copilot", "course"], check=True)
        except subprocess.CalledProcessError as e:
            print(f"Warning: Failed to clone the repository: {e}. Continuing setup anyway...")
        # --- changed: only move into course/notebooks if it actually exists ---
        if pathlib.Path("course/notebooks").exists():
            os.chdir("course/notebooks")
        else:
            print("Warning: course/notebooks not found in the cloned repo. Continuing setup anyway...")
    # 3) keys from Colab Secrets (key icon in the left sidebar): add GOOGLE_API_KEY
    from google.colab import userdata
    for k in ("GOOGLE_API_KEY", "LLM_MODEL", "EMBED_MODEL", "LANGSMITH_API_KEY"):
        try:
            v = userdata.get(k)
            if v: os.environ[k] = v
        except Exception:
            pass
else:
    from dotenv import load_dotenv
    load_dotenv()

# Defaults: Gemini free tier. Override with LLM_MODEL / EMBED_MODEL secrets or .env for any other provider.
LLM_MODEL   = os.environ["LLM_MODEL"] = "google_genai:gemini-3.5-flash"
EMBED_MODEL = os.environ.setdefault("EMBED_MODEL", "google_genai:models/gemini-embedding-001")

if LLM_MODEL.startswith("google_genai"):
    assert os.environ.get("GOOGLE_API_KEY"), "Add GOOGLE_API_KEY in Colab Secrets (key icon) and enable Notebook access, then re-run."

# Free-tier friendly: a client-side rate limiter (~8 requests/min), retries and a timeout on every call.
from langchain_core.rate_limiters import InMemoryRateLimiter
from langchain.chat_models import init_chat_model
LIMITER = InMemoryRateLimiter(requests_per_second=0.14, check_every_n_seconds=0.2, max_bucket_size=4)
model = init_chat_model(LLM_MODEL, temperature=0, rate_limiter=LIMITER, max_retries=3, timeout=60)
print("Colab:", IN_COLAB, "| model:", LLM_MODEL, "| embeddings:", EMBED_MODEL)

Colab: True | model: google_genai:gemini-3.5-flash | embeddings: google_genai:models/gemini-embedding-001


### Step 1 · State

In [16]:
import operator
from typing import Annotated, TypedDict, Literal
from langgraph.graph.message import add_messages

class State(TypedDict):
    topic: str
    plan: list[str]
    notes: Annotated[list[str], operator.add]   # reducer: append
    draft: str
    critique: str
    iterations: int
    messages: Annotated[list, add_messages]     # not used by every node, shown for completeness

### Step 2 · Nodes are functions: state in, partial update out

In [17]:
from pydantic import BaseModel, Field

class Plan(BaseModel):
    questions: list[str] = Field(description="3 research questions, each answerable in one paragraph")

# A stub 'search' tool so the lab runs offline. Replace with search_docs from Lab 3 or a web search tool.
FACTS = {"langgraph": "LangGraph models agents as graphs over typed state with checkpointing.",
         "rag": "RAG retrieves relevant passages and puts them in the prompt.",
         "agents": "An agent is a model that can call tools in a loop until done."}

def plan(state: State):
    p = model.with_structured_output(Plan).invoke(f"Plan a short explainer on: {state['topic']}")
    return {"plan": p.questions, "iterations": 0}

def research(state: State):
    notes = []
    for q in state["plan"]:
        hit = next((v for k, v in FACTS.items() if k in q.lower()), "No stub fact; answer from general knowledge.")
        notes.append(f"Q: {q}\nA: {hit}")
    return {"notes": notes}

def draft(state: State):
    prompt = (f"Write a 120-word explainer on {state['topic']} using these notes:\n" + "\n".join(state["notes"]))
    if state.get("critique"):
        prompt += f"\n\nRevise to address this critique:\n{state['critique']}"
    return {"draft": model.invoke(prompt).content, "iterations": state["iterations"] + 1}

class Critique(BaseModel):
    acceptable: bool
    feedback: str = Field(description="Specific, actionable; empty if acceptable")

def critique(state: State):
    c = model.with_structured_output(Critique).invoke(
        f"You are a strict editor. Is this explainer clear, accurate and under 150 words?\n\n{state['draft']}")
    return {"critique": "" if c.acceptable else c.feedback}

### Step 3 · Wire the graph with a bounded loop

In [20]:
from langgraph.graph import StateGraph, START, END

MAX_ITER = 3

def should_revise(state: State) -> Literal["draft", "done"]:
    """YOUR TURN: return "draft" when there is a non-empty critique AND we are under MAX_ITER; otherwise "done"."""
    if state.get("critique") and state.get("iterations", 0) < MAX_ITER:
        return "draft"
    return "done"

b = StateGraph(State)
b.add_node("plan", plan)
b.add_node("research", research)
b.add_node("draft", draft)
b.add_node("critique", critique)
b.add_edge(START, "plan")
b.add_edge("plan", "research")
b.add_edge("research", "draft")
b.add_edge("draft", "critique")
b.add_conditional_edges("critique", should_revise, {"draft": "draft", "done": END})

print(b.compile().get_graph().draw_mermaid())

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	plan(plan)
	research(research)
	draft(draft)
	critique(critique)
	__end__([<p>__end__</p>]):::last
	__start__ --> plan;
	critique -. &nbsp;done&nbsp; .-> __end__;
	critique -.-> draft;
	draft --> critique;
	plan --> research;
	research --> draft;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



<details><summary>Reference solution for should_revise</summary>

```python
def should_revise(state):
    if state["critique"] and state["iterations"] < MAX_ITER:
        return "draft"
    return "done"
```
</details>

### Step 4 · Compile with a checkpointer and stream

In [21]:
from langgraph.checkpoint.memory import InMemorySaver

graph = b.compile(checkpointer=InMemorySaver())
config = {"configurable": {"thread_id": "lab4-run-1"}}

for update in graph.stream({"topic": "LangGraph for agents"}, config, stream_mode="updates"):
    for node, out in update.items():
        print(f"== {node}: " + ", ".join(f"{k}={str(v)[:60]!r}" for k, v in out.items()))

final = graph.get_state(config).values
print("\nIterations:", final["iterations"])
print(final["draft"])

== plan: plan="['What is LangGraph, and how does its stateful graph archite", iterations='0'
== research: notes="['Q: What is LangGraph, and how does its stateful graph arch"
== draft: draft="[{'type': 'text', 'text': 'Standard linear LLM chains strugg", iterations='1'
== critique: critique=''

Iterations: 1
[{'type': 'text', 'text': 'Standard linear LLM chains struggle to handle complex, iterative reasoning. LangGraph addresses these limits by modeling agents as graphs over typed state with built-in checkpointing. \n\nIn LangGraph, nodes represent discrete agent actions or functions, while edges define the path of execution. By leveraging conditional state updates, the system enables dynamic routing and cyclic multi-agent workflows. Instead of moving purely forward, agents can loop, self-correct, and pass tasks between specialized sub-agents based on the evolving state.\n\nThis stateful design also provides essential features for production environments. Integrated checkpointing guara

### Step 5 · Time travel: inspect history and replay from a checkpoint

In [22]:
history = list(graph.get_state_history(config))
print(len(history), "checkpoints")
for snap in history[:6]:
    print(snap.metadata.get("step"), snap.next, "iterations=", snap.values.get("iterations"))

# Pick the checkpoint right after 'research' and re-run from there with an edited plan.
after_research = next(s for s in reversed(history) if s.next == ("draft",))
edited_cfg = graph.update_state(after_research.config, {"notes": ["A: Always mention checkpointers."]})  # returns the NEW checkpoint's config
replayed = graph.invoke(None, edited_cfg)
print("\nReplayed draft:\n", replayed["draft"][:300])

6 checkpoints
4 () iterations= 1
3 ('critique',) iterations= 1
2 ('draft',) iterations= 0
1 ('research',) iterations= 0
0 ('plan',) iterations= None
-1 ('__start__',) iterations= None

Replayed draft:
 [{'type': 'text', 'text': "LangGraph is a framework that overcomes the limitations of rigid, linear LLM chains by modeling agents as stateful graphs over a shared, typed state. Unlike standard execution chains, it enables complex, cyclic multi-agent workflows. In LangGraph, nodes represent discrete functions or agent actions, edges define control flow, and conditional state updates allow dynamic decision-making. This structure empowers agents to loop, self-correct, and distribute tasks effectively.\n\nFor production applications, LangGraph provides state persistence, human-in-the-loop controls, and fault tolerance through checkpointers. Checkpointers save the graph's state at every step of execution. By leveraging checkpointers, developers can pause workflows for human approval, inspect 

### Step 6 (optional, if time) · The ReAct loop, by hand, in LangGraph (this is what create_agent builds)

In [23]:
from langgraph.prebuilt import ToolNode, tools_condition
from langchain.tools import tool

@tool
def word_count(text: str) -> int:
    """Count words in text."""
    return len(text.split())

class ChatState(TypedDict):
    messages: Annotated[list, add_messages]

llm = model.bind_tools([word_count])
def call_model(state: ChatState):
    return {"messages": [llm.invoke(state["messages"])]}

rb = StateGraph(ChatState)
rb.add_node("model", call_model)
rb.add_node("tools", ToolNode([word_count]))
rb.add_edge(START, "model")
rb.add_conditional_edges("model", tools_condition)   # -> "tools" or END
rb.add_edge("tools", "model")
react = rb.compile()

out = react.invoke({"messages": [{"role": "user", "content": f"How many words are in this draft?\n{final['draft']}"}]})
print(out["messages"][-1].content)

[{'type': 'text', 'text': 'There are **117 words** in this draft.', 'extras': {'signature': 'EpYWCpMWAWkUfRN7EIVxK5hRt0Tbh9tSuI4W3GaStGlhNx56dAcQA0SvIobP0TBiLtOXO7Ec9XaNnS1b9QdzB65GchZDpbZ4mxIqbdXsdD0IyKMjF0BDymJwznVfZL2KhzBTd7DDaVzQGnCeKMJZ2FDb6AY7yeyp/72Sxz7iQwjrTzvKz5dy0ZHivIUJv+wzlkFUbFlEwvWnwKWI7I3ajbNPQg+15gIP4iwSj7OVdF8PDZRRPIbK6h8NC7sOLvIs+rXM023tX5A0a0Af/rmvBTO41pdCKZ04RTGiUjS7ZHqBapJKSRzOaQm7RPmqikttj8LT2eMUik2nzrXbGlvIRK/tQl2B4Sz8wWAhpBcG99lCMrCfno/RUG+rCfFuwbLD/fj1DLkHFMaevmW/uqRNPHqAoRRTCYz9Kl0CL5XxkYMt1ap+bFJeuRqAhDNmqURZyCre0QjqQOLye0PuZGp9VJoxp1MRzWH4bT7Kw00Vu46jqYKPjzZ8T/KTTuZSeou+qXz781+23K/GYql6uXvEn/c/bss0YJuBpPszvVTmAHn1klYv89k/WD4ERmNK9crAEKnm3TB/QyoBpbNx4XGY4JKN4PYdXV87+sqMsU79epxeLOXGoiPkOb3QWPjVTGxNjQX8atfwW0lkadivgTG6d2nTyD/d3fhNrWPJJaaYyoiFq6NuMERdQZunm0rpD76FhzWgyyQRxCGwQkz/ZxNUEGtj9jCrZFD19LFNMfnviRpWurtw03K30+AZfwea/8X2CP2DUUkqO2mI8MpY8UUD0+GM2BdNOav7DQ7YlD3/jES3TUf+IZyVLP1mV9CiyFuB4TlCsgpsjBq0RmeWEwVbo/1saKUT2ZqlqXeDuDw+8s7wbL6AjDe8eAJrtM/c0k4s9wM46V6Q/hv

### Stretch
1. Fan-out: replace `research` with a router that returns `[Send("research_one", {"q": q}) for q in state["plan"]]` and a `research_one` node; notes still accumulate via `operator.add`.
2. Stream tokens: `graph.stream(..., stream_mode="messages")` and print only the `draft` node's tokens.
3. Make `critique` a subgraph with its own state and add it as a node.
4. Async: Colab supports top-level `await`. Run `await graph.ainvoke(...)` for three topics concurrently with `asyncio.gather` and time it against the serial version.

### Homework 4 (due Tue 11/17): capstone skeleton
Your capstone, running end-to-end on one real request, plus 5 eval cases with expected answers. A bounded loop with a caller-set `max_iterations` is a stretch, not required. Capstone proposals are approved by Sat 11/14 office hours.